# FreshCart ELT on Snowflake

This notebook follows the instructor reference: extract sample records to compressed JSONL, load them into Snowflake bronze, transform into silver/dim/ref/gold, and run quality checks.

The original reference notebook is preserved. No outputs are included in this working notebook; run cells with your own Snowflake account to create actual results.

Before connecting, copy `.env.example` to `.env`, enter your own values locally, and select a fresh database name for the baseline run. Never commit `.env`.

## Load project code and connect

Install dependencies from the repository root with `python -m pip install -r requirements.txt`. The notebook reads credentials through environment variables (and loads a local `.env` if `python-dotenv` is installed). It does not display credential values.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'elt').is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from elt.freshcart_pipeline import (
    baseline_snapshot, connect_snowflake, quality_checks, query, run_pipeline,
    run_sql, setup_sql, show,
)

database = os.getenv('SNOWFLAKE_DATABASE', 'FRESHCART')
connection = connect_snowflake()
run_sql(connection, setup_sql(database))
print('Snowflake setup completed for the selected database.')

## B1. Baseline run

For the clean baseline, use a new/empty database in `SNOWFLAKE_DATABASE` before running setup. The expected one-run checks from the assignment are 8 raw orders, 6 clean rows, 2 quarantined orders, 4 deduplicated/gold orders, INR 7,785.00 revenue, 126 loyalty points, and 25% rejected. The following cells display actual Snowflake results; they do not hardcode or simulate those results.

In [ ]:
baseline_run_id = run_pipeline(connection)
baseline = baseline_snapshot(connection)
baseline

In [ ]:
show(connection, "SELECT source, COUNT(*) AS raw_rows FROM bronze.orders_raw GROUP BY source ORDER BY source")

## B2. Record traces

The two APIs use different field names and shapes. `SQL_10_ORDERS_CLEAN` maps web `order_id`, `cust_email`, `order_date`, `amount`, etc. and mobile `orderId`, `customer.email`, `orderDate`, `total.value`, etc. into the same typed columns. `SQL_20_ORDERS_DEDUP` partitions on `order_id` and orders by `updated_at DESC, ingest_seq DESC`, so the later mobile ORD-1003 record with amount 2,100 INR survives.

For ORD-1003, the expected calculation from the supplied data is `amount_inr = 2,100 × 1.0 = 2,100`; Meera's Platinum multiplier is 3, so `floor(2,100 / 100) × 3 = 63` points. This is arithmetic from the sample inputs, not a Snowflake execution result.

For Priya, `dim.customers_scd2` is joined by matching email and `order_date BETWEEN valid_from AND valid_to`. ORD-1001 (Sept 1) uses Silver: `floor(1,250 / 100) × 1 = 12`. ORD-1005 (Sept 3, the Gold effective date) uses Gold: `floor(899 / 100) × 2 = 16`.

In [ ]:
show(connection, "SELECT source, raw, run_id, extracted_at FROM bronze.orders_raw WHERE raw:order_id::STRING = 'ORD-1003' OR raw:orderId::STRING = 'ORD-1003' ORDER BY ingest_seq")

In [ ]:
show(connection, "SELECT order_id, source, order_date, amount, currency, updated_at, reasons FROM silver.orders_checked WHERE order_id = 'ORD-1003' ORDER BY updated_at")

In [ ]:
show(connection, "SELECT order_id, order_date, tier, multiplier, amount, currency, fx_rate, amount_inr, loyalty_points FROM gold.fct_orders WHERE order_id IN ('ORD-1001', 'ORD-1003', 'ORD-1005') ORDER BY order_id")

## B3. Unsupported-currency validation

The working SQL adds `unsupported_currency` for values other than INR/USD. Before the rule, ORD-1006 is rejected for `bad_email` and `bad_pincode`; after the rule, it should also include `unsupported_currency` because its currency is EUR. The order remains quarantined, so the gold total should not change. The quarantine `NOT EXISTS` check keys on source, order ID, and updated timestamp, preventing a second copy of the same rejected version. The query below displays the actual reasons after you rerun the pipeline.

In [ ]:
run_pipeline(connection)
show(connection, "SELECT source, order_id, reasons FROM quarantine.orders WHERE order_id = 'ORD-1006' ORDER BY quarantined_at")

## B4. Arjun's SCD Type 2 change

Prediction before execution: CUST-02 has 2 SCD2 periods; ORD-1002 changes from Silver to Gold because the new effective date (2026-08-15) precedes its order date. At INR 83.20 per USD, `42.50 × 83.20 = 3,536.00`; Gold points are `floor(3,536 / 100) × 2 = 70`. Relative to the baseline 126 points, total gold points are predicted to become 161. Exactly one gold row (ORD-1002) is predicted to have a changed row hash and `published_at`. These are predictions; the next cells show actual results only after Snowflake runs.

In [ ]:
before_published = {row[0]: row[1] for row in query(connection, "SELECT order_id, published_at FROM gold.fct_orders")}
run_pipeline(connection, include_arjun_tier_change=True)
after_published = {row[0]: row[1] for row in query(connection, "SELECT order_id, published_at FROM gold.fct_orders")}
changed_published_at = sum(before_published.get(order_id) != published_at for order_id, published_at in after_published.items())
print('Gold rows whose published_at changed:', changed_published_at)
show(connection, "SELECT cust_id, tier, valid_from, valid_to, is_current FROM dim.customers_scd2 WHERE cust_id = 'CUST-02' ORDER BY valid_from")

In [ ]:
show(connection, "SELECT order_id, tier, multiplier, amount_inr, loyalty_points FROM gold.fct_orders WHERE order_id = 'ORD-1002'")
query(connection, "SELECT COUNT(*), SUM(loyalty_points) FROM gold.fct_orders")

`row_hash` covers the business columns used to build each gold row. On a rerun, `MERGE` matches by `order_id` and updates only when the hash differs, refreshing `published_at` only for changed rows. Unchanged gold rows remain untouched. Bronze is an append-only landing layer, so each ordinary run adds 8 order records even though downstream deduplication and the gold merge prevent duplicate business orders. The optional CRM event adds one CRM raw record on its run.

The sixth check counts future-dated rows in `silver.orders_dedup`. It runs in a separate test database so the intentional failure does not contaminate the baseline or Arjun runs. By default the cell uses `FRESHCART_B5_TEST`; set `SNOWFLAKE_B5_DATABASE` in `.env` if you want another isolated database. It uses the same account credentials and creates/selects that database before running. The test-only mock has no amount, a valid customer and pincode, and a future date, so the future-date check should report `PROBLEM` without creating unrelated missing-FX or customer failures. The cleanup below removes the test-only mock from the isolated database after you capture the actual check output.

In [ ]:
qa_connection = connect_snowflake()
qa_database = os.getenv('SNOWFLAKE_B5_DATABASE', 'FRESHCART_B5_TEST')
run_sql(qa_connection, setup_sql(qa_database))
run_pipeline(qa_connection, include_future_order=True)
qa_checks = quality_checks(qa_connection)
qa_checks['future-dated silver orders']

In [ ]:
run_sql(qa_connection, "DELETE FROM bronze.orders_raw WHERE source = 'web' AND raw:order_id::STRING = 'ORD-2099-01'")
run_sql(qa_connection, "DELETE FROM gold.fct_orders WHERE order_id = 'ORD-2099-01'")
run_pipeline(qa_connection)
print('The future test row was removed from the isolated QA database.')

For the written B5 answer, use your lecture notes: explain why retaining raw source records and transforming in the warehouse suits FreshCart's evolving data and traceability needs, then name one lecture case where transforming before loading is preferable. Record actual Snowflake outputs only after running the cells.

In [ ]:
connection.close()
qa_connection.close()